In [ ]:
# Downsample DTI and PSOCT files in their native space
import subprocess, os
from pathlib import Path


out_folder = (Path.cwd() / "../tests/testdata").resolve()
os.makedirs(out_folder, exist_ok=True)

inp_folder = Path('/Users/Vasilis/Downloads/CMC_results/Moe_fnirt_Cross')
inp_file = inp_folder / 'dtifit_FA.nii.gz'
out_file = out_folder / 'volume_in_DTI.nii.gz'
subprocess.run(['flirt',
                '-in', inp_file,
                '-ref', inp_file,
                '-out', out_file,
                '-applyisoxfm', '3'])
inp_file = [inp_folder / 'Ret_slide_deck.nii.gz',
            inp_folder / 'Ori_slide_deck.nii.gz']
out_file = [out_folder / 'Ret_slidedeck_in_PSOCT.nii.gz',
            out_folder / 'slidedeck_in_PSOCT.nii.gz']
for i, o in zip(inp_file, out_file):
    subprocess.run(['flirt',
                    '-in', i,
                    '-ref', i,
                    '-out', o,
                    '-applyisoxfm', '0.45'])


In [9]:
# Recalculate transformation matrices and warps for downsampled images
os.chdir(out_folder)
!flirt -in 'Ret_slidedeck_in_PSOCT.nii.gz' -ref 'volume_in_DTI.nii.gz' -omat 'PSOCT_to_DTI.mat' -dof 12 -interp 'spline'                                                       
!fnirt --in='Ret_slidedeck_in_PSOCT.nii.gz' --ref='volume_in_DTI.nii.gz' --aff='PSOCT_to_DTI.mat' --fout='PSOCT_to_DTI_warpfield'
!invwarp -w 'PSOCT_to_DTI_warpfield.nii.gz' -o 'DTI_to_PSOCT_warpfield.nii.gz' -r 'Ret_slidedeck_in_PSOCT.nii.gz'                                                              


In [10]:
# Split slidedeck to create PSOCT slices
# (better correspondence to slide indices than downsampling the highres slices)
from fsl.transform import affine
from fsl.data.image import Image
from fsl.transform.flirt import readFlirt, fromFlirt

!fslroi 'slidedeck_in_PSOCT.nii.gz' 'slice_085_in_PSOCT' 0 -1 60 1 0 -1 
!fslroi 'slidedeck_in_PSOCT.nii.gz' 'slice_086_in_PSOCT' 0 -1 59 1 0 -1 


In [11]:
# Create slidedeck to slices mapping
import json

inp_file = Image('slidedeck_in_PSOCT.nii.gz')

slides_dict = {}
for i in range(inp_file.shape[1]):
    idx = inp_file.shape[1] - i
    slides_dict[i] = f'slice_{str(idx).zfill(3)}_in_PSOCT.nii.gz'

with open("slidedeck_slice_mapping.json", "w") as f:
    json.dump({int(k): v for k, v in slides_dict.items()}, f, indent=2)

In [12]:
# Register PSOCT slices to DTI space
mat = readFlirt(out_folder / 'PSOCT_to_DTI.mat')
mat = fromFlirt(mat, Image(out_folder / 'Ret_slidedeck_in_PSOCT.nii.gz'), Image(out_folder / 'volume_in_DTI.nii.gz'), from_='world', to='world')
inp_file = [out_folder / 'slice_085_in_PSOCT.nii.gz',
            out_folder / 'slice_086_in_PSOCT.nii.gz',
            out_folder / 'slidedeck_in_PSOCT.nii.gz']
out_file = [out_folder / 'slice_085_in_DTI.nii.gz',
            out_folder / 'slice_086_in_DTI.nii.gz',
            out_folder / 'slidedeck_in_DTI.nii.gz']
for i, o in zip(inp_file, out_file):
    xform = affine.concat(mat, Image(i).getAffine('voxel', 'world'))
    Image(Image(i).data, xform=xform, header=Image(i).header).save(o)